In [1]:
# =============================================================================
# CS-781 Phase 2 — L-NER with Arch #1, WORD-LEVEL tagging, TECHNIQUE switch
#   TECHNIQUE = "full"  -> full fine-tuning (re-scores Experiment 1 with better decoding)
#   TECHNIQUE = "lorma" -> LoRMA+ (Bihany, Patel, Modi): multiplicative low-rank adaptation
#
# What changed vs Experiment 1 (and why):
#   * WORD-LEVEL tagging. The Phase-1 BPE splits a word such as "19.8.2011" into
#     several tokens, and token-level BIO tagging often labelled only some of
#     them, giving truncated spans (e.g. '15' instead of a full date). Now each
#     whitespace-separated word gets ONE tag. The word is represented by its LAST
#     sub-token, because a causal model has only seen the whole word at that point.
#   * Viterbi decoding over words with BIO transition constraints (compared with argmax).
#   * Error-analysis table (correct / boundary / label confusion / missed / spurious).
#   * LoRMA+ option on the query and value projections, backbone frozen.
#
# LoRMA+ (paper, Sec. 3.2.2): h = ((alpha/r) * B A + I) W0 x, with B = 0 at init so the
# adapted model starts exactly equal to the pretrained one. Only A, B (and the NER
# head) are trained. Paper settings used here: r = 8, alpha = 8, adapters on Wq and Wv.
#
# Run in Kaggle: paste each "# %%" block as its own cell, in order.
# =============================================================================

# %% [1] Imports & HF login
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import json, math, random, time, zipfile, re, itertools
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, hf_hub_download

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# %% [2] Config
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); np.random.seed(SEED)

TECHNIQUE = "full"            # "full" or "lorma"  <-- run once with each

ARTIFACT_PATH = "/kaggle/input/datasets/akashmalgav2/phase2token1lorma"          # tokenizer
# !!! EDIT: same Arch #1 retrain checkpoint you used in Experiment 1
MODEL_CKPT_PATH = "/kaggle/input/datasets/akashmalgav2/phase2-model1-lorma/model_best.pt"

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_TEXT_FILES = ["Phase_2_LNER/Train/fold1-text.json", "Phase_2_LNER/Train/fold2-text.json"]
TRAIN_GOLD_FILES = ["Phase_2_LNER/Train/fold1-gold.json", "Phase_2_LNER/Train/fold2-gold.json"]
LABELS_FILE = "Phase_2_LNER/Train/labels.txt"
TEST_TEXT_FILE = "Phase_2_LNER/Test/fold3-text.json"

HOLDOUT_DOCS = 10             # 0 = final run on all 70 docs (use the epoch count found with holdout)
WORD_POOL = "last"            # which sub-token represents a word: "last" (causal model) or "first"
STRIDE = 512
BATCH_SIZE = 8
EVAL_BATCH = 8
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.06
DROPOUT_FT = 0.1
O_WEIGHT = 1.0

if TECHNIQUE == "full":
    EPOCHS, LR_BODY, LR_HEAD = 6, 5e-5, 5e-4
else:
    EPOCHS, LR_BODY, LR_HEAD = 15, 4e-4, 1e-3       # LoRMA lr 4e-4 as in the paper's RoBERTa setup (Table 14)
LORMA_R, LORMA_ALPHA = 8, 8

OUT_DIR = f"/kaggle/working/cs781_phase2_arch1_{TECHNIQUE}"
os.makedirs(OUT_DIR, exist_ok=True)
BEST_PATH = os.path.join(OUT_DIR, "ner_best.pt")
PRED_PATH = os.path.join(OUT_DIR, "prediction.json")
ZIP_PATH = os.path.join(OUT_DIR, f"submission_phase2_arch1_{TECHNIQUE}.zip")
print("TECHNIQUE:", TECHNIQUE, "| epochs:", EPOCHS)

# %% [3] Load Phase-2 data
def hf_local(path):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=path, repo_type="dataset", token=HF_TOKEN)

LABELS = [x.strip() for x in open(hf_local(LABELS_FILE), encoding="utf-8") if x.strip()]
assert len(LABELS) == 12, LABELS

ID2TAG = {0: ("O", None)}
for i, lab in enumerate(LABELS):
    ID2TAG[1 + 2 * i] = ("B", lab)
    ID2TAG[2 + 2 * i] = ("I", lab)
NUM_TAGS = len(ID2TAG)
B_ID = {lab: 1 + 2 * i for i, lab in enumerate(LABELS)}
I_ID = {lab: 2 + 2 * i for i, lab in enumerate(LABELS)}

def norm_entity(e):
    if isinstance(e, (list, tuple)):
        s, en, lab = e[0], e[1], e[2]
        txt = e[3] if len(e) > 3 else None
    elif isinstance(e, dict):
        def pick(*ks):
            for k in ks:
                if k in e:
                    return e[k]
            return None
        s = pick("start", "start_char", "start_offset", "begin")
        en = pick("end", "end_char", "end_offset")
        lab = pick("label", "type", "entity", "tag")
        txt = pick("text", "entity_text", "span")
    else:
        raise ValueError(f"Unrecognised entity format: {e!r}")
    return int(s), int(en), str(lab), txt

texts, gold = {}, {}
for tf, gf in zip(TRAIN_TEXT_FILES, TRAIN_GOLD_FILES):
    t = json.load(open(hf_local(tf), encoding="utf-8"))
    g = json.load(open(hf_local(gf), encoding="utf-8"))
    for k, v in t.items():
        assert k not in texts, f"duplicate doc id {k}"
        texts[k] = v
    for k, v in g.items():
        if isinstance(v, dict):
            v = v.get("entities") or v.get("annotations") or v.get("labels") or []
        gold[k] = [norm_entity(e) for e in v]
for k in texts:
    gold.setdefault(k, [])
lab_counts = Counter(l for ents in gold.values() for (_, _, l, _) in ents)
print("Docs:", len(texts), "| entities per label:", dict(lab_counts))

# %% [4] Tokenizer (Phase-1 BPE) + tokenization with char offsets AND word ids
with open(f"{ARTIFACT_PATH}/legal_bpe.json", "r", encoding="utf-8") as f:
    _tok = json.load(f)
vocab = _tok["itos"]
merges = [tuple(p) for p in _tok["merges"]]
token_to_id = {tok: i for i, tok in enumerate(vocab)}
merges_rank = {p: i for i, p in enumerate(merges)}
PAD_ID, UNK_ID = token_to_id["<pad>"], token_to_id["<unk>"]
BOS_ID, EOS_ID = token_to_id["<s>"], token_to_id["</s>"]
VOCAB_SIZE = len(vocab)
EOW = "\u0100"
WORD_RE = re.compile(r"\S+")

def word_to_symbols(word):
    return tuple(chr(b) for b in word.encode("utf-8", "ignore")) + (EOW,)

def bpe_encode_word(word):
    symbols = list(word_to_symbols(word))
    while len(symbols) > 1:
        pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
        ranked = [(merges_rank[p], p) for p in pairs if p in merges_rank]
        if not ranked:
            break
        _, best = min(ranked)
        new_symbols, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == best[0] and symbols[i + 1] == best[1]:
                new_symbols.append(best[0] + best[1]); i += 2
            else:
                new_symbols.append(symbols[i]); i += 1
        symbols = new_symbols
    return symbols

def tokenize_with_offsets(text):
    """token ids, per-token (char_start, char_end), per-token word index, and word char ranges."""
    ids, offs, wid, words = [], [], [], []
    for wi, m in enumerate(WORD_RE.finditer(text)):
        word, w0 = m.group(), m.start()
        words.append((w0, m.end()))
        b2c = []
        for ci, ch in enumerate(word):
            for _ in ch.encode("utf-8", "ignore"):
                b2c.append(ci)
        bpos = 0
        for sym in bpe_encode_word(word):
            nb = sum(1 for ch in sym if ch != EOW)
            if nb == 0 or not b2c:
                ce = w0 + len(word)
                offs.append((ce, ce))
            else:
                sb = min(bpos, len(b2c) - 1)
                eb = min(bpos + nb, len(b2c))
                offs.append((w0 + b2c[sb], w0 + b2c[eb - 1] + 1))
            bpos += nb
            ids.append(token_to_id.get(sym, UNK_ID))
            wid.append(wi)
    return ids, offs, wid, words

# %% [5] Word-level records and labels
def encode_doc(text, spans=None):
    ids, offs, wid, words = tokenize_with_offsets(text)
    nz_by_word = [[] for _ in words]
    for t, (a, b) in enumerate(offs):
        if b > a:
            nz_by_word[wid[t]].append(t)
    keep = [k for k, lst in enumerate(nz_by_word) if lst]        # words that own at least one real token
    words_v = [words[k] for k in keep]
    pool = [nz_by_word[k][-1] if WORD_POOL == "last" else nz_by_word[k][0] for k in keep]
    rec = {"ids": ids, "words": words_v, "pool": pool, "len": len(text)}
    if spans is None:
        return rec
    spans = sorted(spans, key=lambda x: (x[0], x[1]))
    char_sid = [-1] * len(text)
    for k, (s, e, lab, _) in enumerate(spans):
        for c in range(max(0, s), min(len(text), e)):
            char_sid[c] = k
    tok_labels = [-100] * len(ids)         # only the pooled sub-token of each word is supervised
    word_labels = []
    span_rng = {}
    prev_sid = -1
    for k, (ws, we) in enumerate(words_v):
        sid = -1
        for c in range(ws, we):
            if char_sid[c] >= 0:
                sid = char_sid[c]
                break
        if sid < 0:
            lab_id = 0
        else:
            lab = spans[sid][2]
            lab_id = B_ID[lab] if sid != prev_sid else I_ID[lab]
            lo, hi = span_rng.get(sid, (ws, we))
            span_rng[sid] = (min(lo, ws), max(hi, we))
        word_labels.append(lab_id)
        tok_labels[pool[k]] = lab_id
        prev_sid = sid
    rec.update(labels=tok_labels, word_labels=word_labels, spans=spans, span_rng=span_rng)
    return rec

recs = {k: encode_doc(texts[k], gold[k]) for k in texts}
print(f"Encoded {len(recs)} docs | tokens: {sum(len(r['ids']) for r in recs.values()):,} | "
      f"words: {sum(len(r['words']) for r in recs.values()):,}")

# %% [6] Word-level oracle: best exact-match rate reachable when predicting whole words
def trim_span(text, s, e, lead, trail):
    while s < e and (text[s].isspace() or text[s] in lead):
        s += 1
    while e > s and (text[e - 1].isspace() or text[e - 1] in trail):
        e -= 1
    return s, e

LEAD_OPTS = ["", "(\"'[\u201c\u2018"]
TRAIL_OPTS = ["", ",;", ",;:", ",;:.", ",;:.)\"'\u201d\u2019]", ",;:)\"'\u201d\u2019]"]
total_spans = sum(len(r["spans"]) for r in recs.values())
results = []
for lead, trail in itertools.product(LEAD_OPTS, TRAIL_OPTS):
    ok = 0
    for k, r in recs.items():
        for sid, (ws, we) in r["span_rng"].items():
            s, e = r["spans"][sid][0], r["spans"][sid][1]
            ok += int(trim_span(texts[k], ws, we, lead, trail) == (s, e))
    results.append((ok / max(1, total_spans), lead, trail))
results.sort(key=lambda x: -x[0])
ORACLE_RATE, TRIM_LEAD, TRIM_TRAIL = results[0]
print(f"WORD-level oracle exact-match ceiling: {ORACLE_RATE:.4f} "
      f"(token-level ceiling was 0.940) | TRIM_LEAD={TRIM_LEAD!r} TRIM_TRAIL={TRIM_TRAIL!r}")

# %% [7] Model: Arch #1 backbone (+ optional LoRMA+) with a tag head
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def build_rope_cache(seq_len, head_dim, device, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.einsum("i,j->ij", t, inv_freq)
    emb = torch.cat((freqs, freqs), dim=-1)
    return emb.cos()[None, None, :, :], emb.sin()[None, None, :, :]

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rope(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class LoRMAPlus(nn.Module):
    """LoRMA+ adapter (pre-multiplication form). Given h = W0 x it returns
         h + (alpha/r) * B(A h)  =  ((alpha/r) B A + I) h  =  ((alpha/r) B A + I) W0 x.
       B starts at 0, so the adapted layer initially equals the frozen one."""
    def __init__(self, d, r, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(r, d))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r))
        self.scale = alpha / r
    def forward(self, h):
        return h + self.scale * F.linear(F.linear(h, self.A), self.B)

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        self.n_head, self.head_dim, self.n_embd = n_head, n_embd // n_head, n_embd
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = dropout
        self.q_ad = None          # LoRMA+ adapters, set by add_lorma()
        self.v_ad = None
    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=-1)            # same layout as view(B,T,3,H,D)
        if self.q_ad is not None:
            q = self.q_ad(q)
            v = self.v_ad(v)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, k, cos[:, :, :T, :], sin[:, :, :T, :])
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                             dropout_p=self.dropout if self.training else 0.0)
        return self.proj(out.transpose(1, 2).contiguous().view(B, T, C))

class SwiGLU(nn.Module):
    def __init__(self, n_embd, mult, dropout):
        super().__init__()
        hidden = ((int(mult * n_embd) + 63) // 64) * 64
        self.w1 = nn.Linear(n_embd, hidden, bias=False)
        self.w2 = nn.Linear(n_embd, hidden, bias=False)
        self.w3 = nn.Linear(hidden, n_embd, bias=False)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.w3(F.silu(self.w1(x)) * self.w2(x)))

class Block(nn.Module):
    def __init__(self, n_embd, n_head, ffn_mult, dropout):
        super().__init__()
        self.ln1 = RMSNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, dropout)
        self.ln2 = RMSNorm(n_embd)
        self.ffn = SwiGLU(n_embd, ffn_mult, dropout)
    def forward(self, x, cos, sin):
        x = x + self.attn(self.ln1(x), cos, sin)
        return x + self.ffn(self.ln2(x))

class LegalLLaMA(nn.Module):
    def __init__(self, vocab_size, n_layer, n_head, n_embd, block_size, ffn_mult, dropout):
        super().__init__()
        self.block_size, self.head_dim = block_size, n_embd // n_head
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, ffn_mult, dropout) for _ in range(n_layer)])
        self.ln_f = RMSNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.tok_emb.weight = self.head.weight
        cos, sin = build_rope_cache(block_size, self.head_dim, device="cpu")
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
    def forward_hidden(self, idx):
        x = self.drop(self.tok_emb(idx))
        cos, sin = self.rope_cos.to(idx.device), self.rope_sin.to(idx.device)
        for blk in self.blocks:
            x = blk(x, cos, sin)
        return self.ln_f(x)

def add_lorma(backbone, r, alpha):
    for blk in backbone.blocks:
        d = blk.attn.n_embd
        blk.attn.q_ad = LoRMAPlus(d, r, alpha)
        blk.attn.v_ad = LoRMAPlus(d, r, alpha)

class NERModel(nn.Module):
    def __init__(self, backbone, n_embd, num_tags, dropout):
        super().__init__()
        self.backbone = backbone
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(n_embd, num_tags)
        nn.init.normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)
    def forward(self, idx):
        return self.classifier(self.drop(self.backbone.forward_hidden(idx)))

ckpt = torch.load(MODEL_CKPT_PATH, map_location="cpu")
cfg = ckpt["config"]
assert cfg["vocab_size"] == VOCAB_SIZE, (cfg["vocab_size"], VOCAB_SIZE)
backbone = LegalLLaMA(cfg["vocab_size"], cfg["n_layer"], cfg["n_head"], cfg["n_embd"],
                      cfg["block_size"], cfg["ffn_mult"], DROPOUT_FT)
backbone.load_state_dict(ckpt["model_state_dict"])
CONTENT = cfg["block_size"] - 1

if TECHNIQUE == "lorma":
    for p in backbone.parameters():
        p.requires_grad = False                    # freeze the pretrained weights W0
    add_lorma(backbone, LORMA_R, LORMA_ALPHA)      # adapters are created trainable
model = NERModel(backbone, cfg["n_embd"], NUM_TAGS, DROPOUT_FT).to(device)

n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Technique {TECHNIQUE}: trainable {n_train/1e6:.3f}M of {n_total/1e6:.1f}M params ({100*n_train/n_total:.2f}%)")

# %% [8] Windows, datasets, split
def make_windows(n, content, stride):
    if n <= content:
        return [(0, n)]
    starts = list(range(0, n - content + 1, stride))
    if starts[-1] + content < n:
        starts.append(n - content)
    return [(s, s + content) for s in starts]

doc_ids = sorted(recs.keys())
random.Random(SEED).shuffle(doc_ids)
val_ids = doc_ids[:HOLDOUT_DOCS]
train_ids = doc_ids[HOLDOUT_DOCS:]
print(f"Train docs: {len(train_ids)} | held-out docs: {len(val_ids)}")

class WindowDataset(Dataset):
    def __init__(self, doc_keys):
        self.items = []
        for k in doc_keys:
            r = recs[k]
            for (s, e) in make_windows(len(r["ids"]), CONTENT, STRIDE):
                x = [BOS_ID] + r["ids"][s:e]
                y = [-100] + r["labels"][s:e]
                pad = CONTENT + 1 - len(x)
                self.items.append((x + [PAD_ID] * pad, y + [-100] * pad))
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        x, y = self.items[i]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

train_ds = WindowDataset(train_ids)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
print("Training windows:", len(train_ds), "| steps/epoch:", len(train_loader))

# %% [9] Inference, word-level Viterbi decoding, strict macro-F1, error analysis
@torch.no_grad()
def predict_logp(net, rec):
    """Per-token tag log-probs. Overlapping windows are merged by LEFT CONTEXT
    (the backbone is causal, so a token is best predicted where it has the most
    tokens before it)."""
    net.eval()
    n = len(rec["ids"])
    wins = make_windows(n, CONTENT, STRIDE)
    store = torch.zeros(n, NUM_TAGS)
    best = torch.full((n,), -1, dtype=torch.long)
    for i in range(0, len(wins), EVAL_BATCH):
        chunk = wins[i:i + EVAL_BATCH]
        batch = torch.full((len(chunk), CONTENT + 1), PAD_ID, dtype=torch.long)
        for r_i, (s, e) in enumerate(chunk):
            batch[r_i, 0] = BOS_ID
            batch[r_i, 1:1 + e - s] = torch.tensor(rec["ids"][s:e], dtype=torch.long)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            logits = net(batch.to(device))
        logp = F.log_softmax(logits.float(), dim=-1).cpu()
        for r_i, (s, e) in enumerate(chunk):
            L = e - s
            pos = torch.arange(L)
            idx = s + pos
            upd = pos > best[idx]
            store[idx[upd]] = logp[r_i, 1:1 + L][upd]
            best[idx[upd]] = pos[upd]
    return store

def build_transitions():
    NEG = -1e9
    trans = np.zeros((NUM_TAGS, NUM_TAGS), dtype=np.float32)
    for a in range(NUM_TAGS):
        for b in range(NUM_TAGS):
            tag_a, ent_a = ID2TAG[a]
            tag_b, ent_b = ID2TAG[b]
            if tag_b == "I" and not (tag_a in ("B", "I") and ent_a == ent_b):
                trans[a, b] = NEG
    start = np.array([NEG if ID2TAG[b][0] == "I" else 0.0 for b in range(NUM_TAGS)], dtype=np.float32)
    return trans, start
TRANS, START = build_transitions()

def viterbi(logp):
    n = logp.shape[0]
    score = START + logp[0]
    back = np.zeros((n, NUM_TAGS), dtype=np.int64)
    for t in range(1, n):
        cand = score[:, None] + TRANS
        back[t] = cand.argmax(0)
        score = cand.max(0) + logp[t]
    path = np.zeros(n, dtype=np.int64)
    path[-1] = score.argmax()
    for t in range(n - 1, 0, -1):
        path[t - 1] = back[t, path[t]]
    return path

def word_tags(logp, rec, mode):
    if not rec["pool"]:
        return np.zeros(0, dtype=np.int64)
    lp = logp[rec["pool"]].numpy()          # one row of tag log-probs per word
    return lp.argmax(1) if mode == "argmax" else viterbi(lp)

def decode_spans(text, wtags, rec):
    spans, cur = [], None
    for k, tid in enumerate(wtags):
        ws, we = rec["words"][k]
        tid = int(tid)
        if tid == 0:
            if cur:
                spans.append(cur); cur = None
            continue
        tag, ent = ID2TAG[tid]
        if cur is not None and tag == "I" and cur[2] == ent:
            cur[1] = we
        else:
            if cur:
                spans.append(cur)
            cur = [ws, we, ent]
    if cur:
        spans.append(cur)
    out = set()
    for s, e, ent in spans:
        s2, e2 = trim_span(text, s, e, TRIM_LEAD, TRIM_TRAIL)
        if e2 > s2:
            out.add((s2, e2, ent))
    return sorted(out)

def strict_macro(truth, pred):
    stats = {l: [0, 0, 0] for l in LABELS}
    for d in truth:
        g, p = set(truth[d]), set(pred.get(d, []))
        for l in LABELS:
            gs = {x for x in g if x[2] == l}
            ps = {x for x in p if x[2] == l}
            stats[l][0] += len(gs & ps); stats[l][1] += len(ps - gs); stats[l][2] += len(gs - ps)
    P, R, F_, per = [], [], [], {}
    for l in LABELS:
        tp, fp, fn = stats[l]
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
        P.append(pr); R.append(rc); F_.append(f); per[l] = (pr, rc, f)
    return sum(F_) / len(F_), sum(P) / len(P), sum(R) / len(R), per

def evaluate_heldout(net, keys):
    truth = {k: sorted({(s, e, l) for (s, e, l, _) in gold[k]}) for k in keys}
    preds = {"argmax": {}, "viterbi": {}}
    for k in keys:
        logp = predict_logp(net, recs[k])
        for mode in preds:
            preds[mode][k] = decode_spans(texts[k], word_tags(logp, recs[k], mode), recs[k])
    return {m: strict_macro(truth, preds[m]) for m in preds}, truth, preds

def error_analysis(truth, pred):
    cats = {l: Counter() for l in LABELS}
    spurious = Counter()
    for d in truth:
        g, p = truth[d], pred[d]
        pset, gset = set(p), set(g)
        for (s, e, l) in g:
            if (s, e, l) in pset:
                cats[l]["correct"] += 1
            elif any(pl == l and ps < e and pe > s for ps, pe, pl in p):
                cats[l]["boundary"] += 1
            elif any(pl != l and ps < e and pe > s for ps, pe, pl in p):
                cats[l]["label_confusion"] += 1
            else:
                cats[l]["missed"] += 1
        for (ps, pe, pl) in p:
            if (ps, pe, pl) not in gset and not any(gs < pe and ge > ps for gs, ge, gl in g):
                spurious[pl] += 1
    print(f"{'label':10s} {'gold':>5s} {'correct':>8s} {'boundary':>9s} {'confused':>9s} {'missed':>7s} {'spurious':>9s}")
    for l in LABELS:
        c = cats[l]
        print(f"{l:10s} {sum(c.values()):5d} {c['correct']:8d} {c['boundary']:9d} {c['label_confusion']:9d} {c['missed']:7d} {spurious[l]:9d}")

# %% [10] Training
class_w = torch.ones(NUM_TAGS)
class_w[0] = O_WEIGHT
class_w = class_w.to(device)

head_params = list(model.classifier.parameters())
head_ids = {id(p) for p in head_params}
body_params = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
optimizer = torch.optim.AdamW(
    [{"params": body_params, "lr": LR_BODY, "base": LR_BODY},
     {"params": head_params, "lr": LR_HEAD, "base": LR_HEAD}],
    weight_decay=WEIGHT_DECAY, betas=(0.9, 0.98))
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

total_steps = EPOCHS * len(train_loader)
warm = max(1, int(WARMUP_FRAC * total_steps))
def lr_factor(step):
    if step < warm:
        return step / warm
    prog = (step - warm) / max(1, total_steps - warm)
    return max(0.02, 0.5 * (1 + math.cos(math.pi * prog)))

history, best_f1, step, t0 = [], -1.0, 0, time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    run_loss, n_b = 0.0, 0
    for x, y in train_loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        for g in optimizer.param_groups:
            g["lr"] = g["base"] * lr_factor(step)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            logits = model(x)
        loss = F.cross_entropy(logits.float().view(-1, NUM_TAGS), y.view(-1), weight=class_w, ignore_index=-100)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        scaler.step(optimizer)
        scaler.update()
        run_loss += loss.item(); n_b += 1; step += 1
    rec_h = {"epoch": epoch, "train_loss": run_loss / max(1, n_b)}
    if val_ids:
        out, _, _ = evaluate_heldout(model, val_ids)
        rec_h.update(f1_argmax=out["argmax"][0], f1_viterbi=out["viterbi"][0])
        print(f"[epoch {epoch}] loss {rec_h['train_loss']:.4f} | held-out strict macro-F1: "
              f"argmax {out['argmax'][0]:.4f} | viterbi {out['viterbi'][0]:.4f} "
              f"(P {out['viterbi'][1]:.4f} R {out['viterbi'][2]:.4f}) | {(time.time()-t0)/60:.1f}m")
        if out["viterbi"][0] > best_f1:
            best_f1 = out["viterbi"][0]
            torch.save({"model": model.state_dict(), "epoch": epoch, "val_f1": best_f1}, BEST_PATH)
            print("  -> new best checkpoint saved")
    else:
        print(f"[epoch {epoch}] loss {rec_h['train_loss']:.4f} (no held-out; final run on all docs)")
        torch.save({"model": model.state_dict(), "epoch": epoch, "val_f1": None}, BEST_PATH)
    history.append(rec_h)
json.dump(history, open(os.path.join(OUT_DIR, "history.json"), "w"), indent=2)

# %% [11] Per-label scores + error analysis
best = torch.load(BEST_PATH, map_location="cpu")
model.load_state_dict(best["model"])
DECODE_MODE = "viterbi"
if val_ids:
    out, val_truth, val_preds = evaluate_heldout(model, val_ids)
    DECODE_MODE = max(out, key=lambda m: out[m][0])
    print(f"Best epoch {best['epoch']} | argmax F1 {out['argmax'][0]:.4f} | viterbi F1 {out['viterbi'][0]:.4f} -> using {DECODE_MODE}")
    per = out[DECODE_MODE][3]
    for l in LABELS:
        print(f"  {l:10s} P={per[l][0]:.3f} R={per[l][1]:.3f} F1={per[l][2]:.3f}  (gold n={lab_counts.get(l,0)})")
    print()
    error_analysis(val_truth, val_preds[DECODE_MODE])
    json.dump({k: [list(x) for x in v] for k, v in val_truth.items()}, open(os.path.join(OUT_DIR, "val_truth.json"), "w"))
    json.dump({k: [list(x) for x in v] for k, v in val_preds[DECODE_MODE].items()}, open(os.path.join(OUT_DIR, "val_pred.json"), "w"))

# %% [12] Predict the test set (fold3) -> prediction.json -> zip
test_texts = json.load(open(hf_local(TEST_TEXT_FILE), encoding="utf-8"))
print("Test docs:", len(test_texts), "| decode mode:", DECODE_MODE)
test_pred = {}
t0 = time.time()
for i, (k, text) in enumerate(test_texts.items()):
    rec = encode_doc(text)
    spans = decode_spans(text, word_tags(predict_logp(model, rec), rec, DECODE_MODE), rec)
    test_pred[k] = [[s, e, l] for (s, e, l) in spans]
    if (i + 1) % 10 == 0:
        print(f"  predicted {i+1}/{len(test_texts)} | {(time.time()-t0)/60:.1f}m")

assert set(test_pred) == set(test_texts), "ID mismatch"
for k, spans in test_pred.items():
    n = len(test_texts[k])
    for s, e, l in spans:
        assert 0 <= s < e <= n and l in LABELS, (k, s, e, l)
tot = sum(len(v) for v in test_pred.values())
print("Total predicted entities:", tot, "| per doc:", round(tot / len(test_pred), 1), "(training gold averages ~120 per doc)")
ex_k = next(k for k, v in test_pred.items() if v)
print("Sample:", [(s, e, l, test_texts[ex_k][s:e]) for s, e, l in test_pred[ex_k][:5]])

with open(PRED_PATH, "w", encoding="utf-8") as f:
    json.dump(test_pred, f)
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(PRED_PATH, arcname="prediction.json")
print("Codabench submission ready:", ZIP_PATH)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Device: cuda
TECHNIQUE: full | epochs: 6


labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

Docs: 70 | entities per label: {'RESP': 381, 'DATE': 1575, 'COURT': 1013, 'WIT': 187, 'APP': 407, 'STAT': 1260, 'A.COUNSEL': 195, 'PREC': 1183, 'JUDGE': 245, 'R.COUNSEL': 184, 'CASENO': 760, 'AUTH': 1003}
Encoded 70 docs | tokens: 421,043 | words: 372,576
WORD-level oracle exact-match ceiling: 0.9048 (token-level ceiling was 0.940) | TRIM_LEAD='("\'[“‘' TRIM_TRAIL=',;:.)"\'”’]'
Technique full: trainable 66.237M of 66.2M params (100.00%)
Train docs: 60 | held-out docs: 10
Training windows: 658 | steps/epoch: 83
[epoch 1] loss 0.7364 | held-out strict macro-F1: argmax 0.2127 | viterbi 0.2682 (P 0.3370 R 0.2568) | 0.6m
  -> new best checkpoint saved
[epoch 2] loss 0.0873 | held-out strict macro-F1: argmax 0.2908 | viterbi 0.3744 (P 0.3835 R 0.3728) | 1.2m
  -> new best checkpoint saved
[epoch 3] loss 0.0621 | held-out strict macro-F1: argmax 0.3131 | viterbi 0.3894 (P 0.3800 R 0.4035) | 1.8m
  -> new best checkpoint saved
[epoch 4] loss 0.0495 | held-out strict macro-F1: argmax 0.3212 | v

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

Test docs: 35 | decode mode: viterbi
  predicted 10/35 | 0.0m
  predicted 20/35 | 0.1m
  predicted 30/35 | 0.2m
Total predicted entities: 4156 | per doc: 118.7 (training gold averages ~120 per doc)
Sample: [(18, 40, 'COURT', 'SUPREME COURT OF INDIA'), (70, 93, 'CASENO', 'CIVIL APPEAL NO.6580 OF'), (122, 162, 'CASENO', 'LEAVE PETITION (CIVIL) NO. 15449 OF 2004'), (163, 182, 'APP', 'ESSCO FABS PVT. LTD'), (213, 229, 'RESP', 'STATE OF HARYANA')]
Codabench submission ready: /kaggle/working/cs781_phase2_arch1_full/submission_phase2_arch1_full.zip


In [2]:
# %% [13] Package all LoRMA+ output files into one ZIP

import os
import zipfile

assert TECHNIQUE == "full", (
    f"Expected LoRMA+ run, but TECHNIQUE={TECHNIQUE!r}"
)

FINAL_ZIP = "/kaggle/working/submission_phase2_arch1_full_all_files.zip"

# Files generated by the notebook
files_to_include = [
    PRED_PATH,                                      # prediction.json
    BEST_PATH,                                      # ner_best.pt
    os.path.join(OUT_DIR, "history.json"),          # training history
    os.path.join(OUT_DIR, "val_truth.json"),        # validation gold spans
    os.path.join(OUT_DIR, "val_pred.json"),         # validation predictions
]

# Include other files saved in the output directory, if any
for filename in os.listdir(OUT_DIR):
    path = os.path.join(OUT_DIR, filename)
    if os.path.isfile(path) and path not in files_to_include:
        files_to_include.append(path)

# Keep only files that actually exist
files_to_include = list(dict.fromkeys(
    path for path in files_to_include if os.path.isfile(path)
))

assert os.path.isfile(PRED_PATH), "prediction.json is missing!"
assert os.path.isfile(BEST_PATH), "Best LoRMA+ checkpoint is missing!"

with zipfile.ZipFile(
    FINAL_ZIP, "w", compression=zipfile.ZIP_DEFLATED
) as zf:
    for path in files_to_include:
        zf.write(path, arcname=os.path.basename(path))

print("LoRMA+ ZIP created:", FINAL_ZIP)
print(f"Files included: {len(files_to_include)}")
print(f"ZIP size: {os.path.getsize(FINAL_ZIP) / (1024**2):.2f} MB")

print("\nZIP contents:")
with zipfile.ZipFile(FINAL_ZIP, "r") as zf:
    for name in zf.namelist():
        info = zf.getinfo(name)
        print(f"  {name:25s} {info.file_size / (1024**2):8.2f} MB")

LoRMA+ ZIP created: /kaggle/working/submission_phase2_arch1_full_all_files.zip
Files included: 6
ZIP size: 234.33 MB

ZIP contents:
  prediction.json               0.09 MB
  ner_best.pt                 252.71 MB
  history.json                  0.00 MB
  val_truth.json                0.03 MB
  val_pred.json                 0.03 MB
  submission_phase2_arch1_full.zip     0.03 MB
